In [11]:
import os
import numpy as np
import pandas as pd

import rocks
rocks.set_log_level("error")
import time as t

import requests
import wget


In [45]:
# DAMIT
path_damit = os.path.join('..','..','data','damit')
url = 'https://astro.troja.mff.cuni.cz/projects/damit/generated_files/open/AsteroidModel/'
url_lc = 'https://astro.troja.mff.cuni.cz/projects/damit/LightCurves/lcRef/'

# Read data

In [20]:
for f in ['asteroids', 'asteroid_models', 'references', 'asteroid_models_references']:
    file = os.path.join( path_damit, f'damit-{f}.csv')
    print(file)
    if not os.path.exists(file):
        fname = wget.download(f'https://astro.troja.mff.cuni.cz/projects/damit/exports/table/{f}')
        os.rename(fname, file)

../../data/damit/damit-asteroids.csv
../../data/damit/damit-asteroid_models.csv
../../data/damit/damit-references.csv
../../data/damit/damit-asteroid_models_references.csv


In [21]:
# Read DAMIT Tables
ssos = pd.read_csv(os.path.join( path_damit, 'damit-asteroids.csv'))
models = pd.read_csv(os.path.join( path_damit, 'damit-asteroid_models.csv'))
refs = pd.read_csv(os.path.join( path_damit, 'damit-asteroid_models_references.csv'))
bibs = pd.read_csv(os.path.join( path_damit, 'damit-references.csv'))

In [22]:
# Select columns for ASTEROIDS
ssos = ssos[ ['id', 'number', 'name', 'designation'] ]
ssos.columns = ['asteroid_id', 'number', 'name', 'designation']

In [23]:
# Select columns for MODELS
models = models[ [ 'id', 'asteroid_id', 'lambda', 'beta', 'period' ] ]
models.columns = [ 'model_id', 'asteroid_id', 'lambda', 'beta', 'period' ]

In [24]:
# Select columns for REFERENCES
refs = refs[ ['asteroid_model_id','reference_id'] ]
refs.columns = ['model_id','reference_id']

In [25]:
# Select columns for BIBLIOGRAPHY
bibs = bibs[ ['id', 'bibcode'] ]
bibs.columns = ['reference_id', 'bibcode']

In [26]:
# Merge everything
data = models.merge( ssos, on='asteroid_id')
data = data.merge( refs, on='model_id')
data = data.merge( bibs, on='reference_id')
data

,model_id,asteroid_id,lambda,beta,period,number,name,designation,reference_id,bibcode
0,101,101,35.0,-12.0,7.813230,2.0,Pallas,NaN,106,2003icar..164..346t
1,101,101,35.0,-12.0,7.813230,2.0,Pallas,NaN,139,2011icar..214..652d
2,102,101,32.0,-11.0,7.813220,2.0,Pallas,NaN,169,2017a&a...601a.114h
3,102,101,32.0,-11.0,7.813220,2.0,Pallas,NaN,132,2010icar..205..460c
4,103,102,103.0,27.0,7.209531,3.0,Juno,NaN,103,2002icar..159..369k
...,...,...,...,...,...,...,...,...,...,...
16314,16301,2202,34.0,39.0,16.856900,806.0,Gyldenia,NaN,667,NaN
16315,16302,10862,318.0,60.0,4.463668,138852.0,NaN,NaN,668,2024a&a...682a..93d
16316,16303,10863,260.0,-60.0,7.664354,85989.0,NaN,NaN,668,2024a&a...682a..93d
16317,16304,10864,230.0,36.0,35.984000,357.0,Ninina,NaN,674,2024mpbu...51..100f


In [27]:
data[ data.number==135 ]

,model_id,asteroid_id,lambda,beta,period,number,name,designation,reference_id,bibcode
119,162,142,272.0,52.0,8.4006,135.0,Hertha,NaN,106,2003icar..164..346t
120,162,142,272.0,52.0,8.4006,135.0,Hertha,NaN,127,2009mpbu...36...98t
1846,1799,142,276.0,53.0,8.4006,135.0,Hertha,NaN,169,2017a&a...601a.114h


# Download spins and shapes

In [50]:
for i, r in data.iterrows():
 
    if i%100==0:
        print(i, r['number'], r['name'], r['model_id'])

    # Shape model
    url_shape = f"{url}{r['model_id']}/shape.obj"
    file_shape = os.path.join(path_damit, f"{r['model_id']:d}.obj")
    
    # Spin file
    url_spin = f"{url}{r['model_id']}/spin.txt"
    file_spin = os.path.join(path_damit, f"{r['model_id']:d}.spin")

    # LCRef
    url_lcref = f"{url_lc}{r['asteroid_id']}/A{r['asteroid_id']}.lc.ref.txt"
    file_lcref = os.path.join(path_damit, f"{r['asteroid_id']:d}.lcref")

    # Download files
    for u, f in zip([url_shape, url_spin, url_lcref], [file_shape, file_spin, file_lcref]):
        if not os.path.exists(f):
            response = requests.get(u)
            with open(f, mode="wb") as tok:
                tok.write(response.content)

    # if i==4:
    #     break


0 2.0 Pallas 101
100 110.0 Lydia 152
200 423.0 Diotima 217
300 64.0 Angelina 288
400 606.0 Brangane 372
500 47035.0 nan 460
600 147.0 Protogeneia 557
700 874.0 Rotraut 653
800 3492.0 Petra-Pepi 750
900 639.0 Latona 855
1000 762.0 Pulcova 954
1100 3725.0 Valsecchi 1055
1200 1278.0 Kenya 1162
1300 1075.0 Helina 1266
1400 1768.0 Appenzella 1358
1500 27225.0 nan 1453
1600 4089.0 Galbraith 1553
1700 769.0 Tatjana 1653
1800 1641.0 Tana 1754
1900 23.0 Thalia 1857
2000 12088.0 Macalintal 1962
2100 34290.0 nan 2062
2200 30895.0 nan 2162
2300 120796.0 nan 2264
2400 10559.0 Yukihisa 2364
2500 28343.0 nan 2464
2600 59150.0 nan 2564
2700 33181.0 Aalokpatwa 2664
2800 54391.0 nan 2764
2900 134361.0 nan 2864
3000 27471.0 nan 2964
3100 11174.0 Carandrews 3064
3200 18156.0 Kamisaibara 3167
3300 21842.0 nan 3267
3400 7571.0 Weisse Rose 3367
3500 22097.0 nan 3467
3600 3140.0 Stellafane 3573
3700 8981.0 nan 3673
3800 7556.0 Perinaldo 3773
3900 1912.0 Anubis 3873
4000 11658.0 nan 3975
4100 9132.0 Walterande

In [ ]:
# run the following command in damit directory to get abc file
echo "model_id,a,b,c,J2">abc.csv;for k in $(ls *obj) ; do id=${k%%.obj}; l=$(obj2ver.sh ${id}.obj|inertia -a);echo "${id},${l}">> abc.csv;done

# and that one for the info on asteroid_id,N_LC_total,N_sparse
echo "asteroid_id,N,N_SP">lc_summary.csv;for k in $(ls *lcref) ; do id=${k%%.lcref}; n_ep=$(cat $k|wc -l);n_sp=$(grep '\-\-' $k|wc -l); echo "${id},${n_ep},${n_sp}">> lc_summary.csv;done